# LT separation bin helper: linked phase-space dashboard

Use the compact dashboard to compare one or several epsilon settings. Click four corners on **either** $W$–$Q^2$ or $x_B$–$Q^2$ to make a diamond cut. Its projection, selected events, $t'$/$Q^2$/$x_B$ distributions, and $t'$–$\phi$ occupancy update together. Choose a corner number and click again to move it. Outer limits and uniform/quantile bin modes are adjustable in the same view.

All edge selection and diagnostic plots use **data only**. $\phi$ bins stay uniform. The later SIMC section is for the reference setting only. No acceptance, yield ratio, or cross section is calculated.

## Running the notebook

Use `/group/nps/singhav/software/python/bin/python` as the Jupyter kernel. Plotly, ipywidgets, and anywidget are installed there. Restart that kernel after installing or upgrading packages, then run all cells from top to bottom.

The linked view appears directly below the first $t'$ candidate table. Its panels are arranged together in two rows. **Click four corners in order around the desired region on either 2D panel**; the other 2D panel and all one-dimensional views update immediately. Each accepted click appears as a yellow numbered marker on both 2D plots; the status line confirms the corner and coordinates. Select a corner number to replace one vertex, or use Undo/Clear. The controls update plots when released. The “Save” box gives exact configuration values to copy into the configuration cell for a future run. Changes in the dashboard also update the current kernel's selection for cells executed afterward; rerun later cells after changing the dashboard.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.path import Path as PolygonPath
import uproot
from IPython.display import Markdown, display
try:
    import plotly.graph_objects as go
except ImportError as exc:
    raise ImportError("Install Plotly in this notebook kernel with %pip install plotly, then restart the kernel") from exc

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold"})
pd.set_option("display.max_colwidth", 140)

## Configuration

`KINEMATICS` accepts one or several keys from `config/nps_simulation_kinematics*.csv`. The 407 cm row is selected by its own key, `KinC_x36_5_407`. The CSV supplies the beam energy used for the epsilon diagnostic. `REFERENCE_KINEMATIC` selects the setting for the existing SIMC representative-coordinate section.

Use `Q2_LIMITS`, `XB_LIMITS`, and `TPRIME_LIMITS` as outer edges. Optional `DIAMOND_XB_Q2_VERTICES` restricts the common phase space. Candidate modes are `uniform`, `count_quantile`, and `yield_quantile`; `MANUAL_TPRIME_EDGES` and `MANUAL_Q2_EDGES` supply manual arrays; `XB_EDGES_BY_Q2` supplies manual arrays within each Q2 bin. `XB_EDGES_BY_Q2` can override the global xB mode with a separate manual array for every Q2 bin. These are selection aids; review coverage for **each** setting before freezing edges.

In [2]:
REPO_ROOT = Path("/w/hallc-scshelf2102/nps/singhav/nps_analysis/pi0_analysis/root_analysis_env_main")
DATA_FILES = {
    "KinC_x36_4": REPO_ROOT / "output/KinC_x36_4/root/combined_branches_LH2.root",
    "KinC_x36_5_407": REPO_ROOT / "output/KinC_x36_5_407/KinC_x36_5/root/combined_branches_LH2.root",
}
KINEMATICS = ["KinC_x36_4", "KinC_x36_5_407"]  # Or ["KinC_x36_4"]
REFERENCE_KINEMATIC = "KinC_x36_5_407"
SIM_FILES = {
    "KinC_x36_5_407": Path("/volatile/hallc/nps/singhav/nps_smearing/smear_x36_5_407/smearing_output/KinC_x36_5_407/root/simc_pi0_analysis_output_smeared.root"),
}
SIM_FILE = SIM_FILES.get(REFERENCE_KINEMATIC)
DATA_TREE = "physics"
SIM_TREE = "simulation"

N_TPRIME = 4
N_PHI = 18
TPRIME_LIMITS = (-0.7, 0.0)     # GeV^2
Q2_LIMITS = (3.0, 5.0)           # GeV^2
XB_LIMITS = (0.25, 0.455)
MMISS_LIMITS = (0.80, 1.10)      # GeV
DIAMOND_XB_Q2_VERTICES = None   # Example: [(0.30, 3.2), (0.39, 4.4), (0.45, 4.8), (0.34, 3.6)]

TPRIME_EDGE_MODE = "count_quantile"
MANUAL_TPRIME_EDGES = None
Q2_EDGE_MODE = "count_quantile"
N_Q2 = 2
MANUAL_Q2_EDGES = None
XB_EDGE_MODE = "count_quantile"
N_XB_PER_Q2 = 2
XB_EDGES_BY_Q2 = None
PHI_EDGES = np.linspace(0.0, 2.0 * np.pi, N_PHI + 1)

# Display controls: the occupancy map uses selected events; optional dots are sampled.
OVERLAP_BINS = (100, 90)
MIN_EVENTS_PER_CELL = 3
MAX_SCATTER_PER_SETTING = 12000

phi_cpp_edges = ["0.0"] + [f"{2 * i}*TMath::Pi()/{N_PHI}" for i in range(1, N_PHI)] + ["2*TMath::Pi()"]
print("std::vector<double> phi_bin_edges = {")
phi_cpp_rows = [phi_cpp_edges[:4]] + [phi_cpp_edges[start:start + 3] for start in range(4, len(phi_cpp_edges), 3)]
for index, row in enumerate(phi_cpp_rows):
    suffix = "," if index + 1 < len(phi_cpp_rows) else ""
    print("    " + ", ".join(row) + suffix)
print("};")

PREVIEW_BIN_COUNTS = (1, 2, 3, 4)
PREVIEW_N_Q2 = 2
PREVIEW_N_XB_PER_Q2 = 2
PROTON_MASS_GEV = 0.9382720813
WRITE_EXPORT = False
EXPORT_PATH = REPO_ROOT / "src/xsec_extract/xsec_bin_selection.json"

if not KINEMATICS or len(KINEMATICS) != len(set(KINEMATICS)):
    raise ValueError("KINEMATICS must contain distinct settings")
if REFERENCE_KINEMATIC not in KINEMATICS:
    raise ValueError("REFERENCE_KINEMATIC must be one of KINEMATICS")
if not all(lo < hi for lo, hi in (TPRIME_LIMITS, Q2_LIMITS, XB_LIMITS, MMISS_LIMITS)):
    raise ValueError("Every outer limit must increase")
if DIAMOND_XB_Q2_VERTICES is not None and len(DIAMOND_XB_Q2_VERTICES) != 4:
    raise ValueError("A diamond polygon needs exactly four (xB, Q2) vertices")

std::vector<double> phi_bin_edges = {
    0.0, 2*TMath::Pi()/18, 4*TMath::Pi()/18, 6*TMath::Pi()/18,
    8*TMath::Pi()/18, 10*TMath::Pi()/18, 12*TMath::Pi()/18,
    14*TMath::Pi()/18, 16*TMath::Pi()/18, 18*TMath::Pi()/18,
    20*TMath::Pi()/18, 22*TMath::Pi()/18, 24*TMath::Pi()/18,
    26*TMath::Pi()/18, 28*TMath::Pi()/18, 30*TMath::Pi()/18,
    32*TMath::Pi()/18, 34*TMath::Pi()/18, 2*TMath::Pi()
};


## 1. Load and select data

Each ROOT file is opened independently. The missing-mass and $t'$ cuts define the **coverage sample** used in the 2D comparison. The $Q^2$, $x_B$, and optional diamond cuts then define the **binning sample**. This distinction keeps the full coverage visible while showing the proposed common region. The display weight follows the existing convention within each setting: `pi0_weight * scale * (run charge / setting total charge)`. Count quantiles ignore weights.

In [3]:
DATA_BRANCHES = [
    "Q2", "xB", "W", "t", "tmin", "phi", "mmiss_all",
    "pi0_weight", "scale", "charge_uC", "run_number",
]

def open_checked_tree(path, tree_name, required_branches):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    root_file = uproot.open(path)
    if tree_name not in root_file:
        raise KeyError(f"Tree {tree_name!r} is absent from {path}")
    tree = root_file[tree_name]
    missing = sorted(set(required_branches) - set(tree.keys()))
    if missing:
        raise KeyError(f"Missing branches in {path}: {missing}")
    return root_file, tree

def tree_frame(tree, branches):
    arrays = tree.arrays(branches, library="np")
    return pd.DataFrame({name: arrays[name] for name in branches})

# Require exactly one matching simulation-kinematics row per setting.
kinematics_tables = []
for csv_path in sorted((REPO_ROOT / "config").glob("nps_simulation_kinematics*.csv")):
    table = pd.read_csv(csv_path, skipinitialspace=True)
    table.columns = table.columns.str.strip()
    table["kin_old"] = table["kin_old"].astype(str).str.strip()
    table["target"] = table["target"].astype(str).str.strip()
    table["csv_source"] = str(csv_path)
    kinematics_tables.append(table)
if not kinematics_tables:
    raise FileNotFoundError("No config/nps_simulation_kinematics*.csv files found")
kinematics_table = pd.concat(kinematics_tables, ignore_index=True)
settings_metadata = {}
for setting in KINEMATICS:
    if setting not in DATA_FILES:
        raise KeyError(f"Add a DATA_FILES path for {setting}")
    matches = kinematics_table.loc[(kinematics_table["kin_old"] == setting) & (kinematics_table["target"] == "LH2")]
    if len(matches) != 1:
        raise ValueError(f"Expected one LH2 kinematics CSV row for {setting}; found {len(matches)}")
    settings_metadata[setting] = matches.iloc[0]
display(pd.DataFrame(settings_metadata).T[["new_kin", "ebeam_gev", "hms_p_gev", "hms_theta_deg", "nps_theta_deg", "nps_target_distance_cm", "csv_source"]])

def diamond_mask(frame):
    if DIAMOND_XB_Q2_VERTICES is None:
        return np.ones(len(frame), dtype=bool)
    vertices = np.asarray(DIAMOND_XB_Q2_VERTICES, dtype=float)
    polygon = PolygonPath(np.vstack([vertices, vertices[0]]), closed=True)
    return polygon.contains_points(frame[["xB", "Q2"]].to_numpy(), radius=1e-9)

all_data, coverage_frames, selected_frames, summary_rows = {}, [], [], []
for setting in KINEMATICS:
    root_file, tree = open_checked_tree(DATA_FILES[setting], DATA_TREE, DATA_BRANCHES)
    frame = tree_frame(tree, DATA_BRANCHES)
    root_file.close()
    frame["setting"] = setting
    frame["tprime"] = frame["t"] - frame["tmin"]
    frame["phi_wrapped"] = np.mod(frame["phi"], 2.0 * np.pi)
    valid_charge = np.isfinite(frame["charge_uC"]) & (frame["charge_uC"] > 0.0)
    run_charges = frame.loc[valid_charge, ["run_number", "charge_uC"]].drop_duplicates("run_number")
    total_charge_uC = run_charges["charge_uC"].sum()
    if not np.isfinite(total_charge_uC) or total_charge_uC <= 0.0:
        raise ValueError(f"{setting} has no valid positive total run charge")
    frame["data_weight"] = frame["pi0_weight"] * frame["scale"] * np.where(valid_charge, frame["charge_uC"] / total_charge_uC, 1.0)
    ebeam = float(settings_metadata[setting]["ebeam_gev"])
    nu = frame["Q2"] / (2.0 * PROTON_MASS_GEV * frame["xB"])
    eprime = ebeam - nu
    tan2 = frame["Q2"] / (4.0 * ebeam * eprime - frame["Q2"])
    frame["epsilon"] = np.where((eprime > 0) & (tan2 > 0), 1.0 / (1.0 + 2.0 * (1.0 + nu**2 / frame["Q2"]) * tan2), np.nan)
    finite = np.logical_and.reduce([np.isfinite(frame[name]) for name in ["Q2", "xB", "W", "tprime", "phi_wrapped", "mmiss_all", "data_weight"]])
    coverage = frame.loc[finite & frame["mmiss_all"].between(*MMISS_LIMITS, inclusive="both") & frame["tprime"].between(*TPRIME_LIMITS, inclusive="both")].copy()
    chosen = coverage.loc[coverage["Q2"].between(*Q2_LIMITS, inclusive="both") & coverage["xB"].between(*XB_LIMITS, inclusive="both") & diamond_mask(coverage)].copy()
    all_data[setting] = frame
    coverage_frames.append(coverage)
    selected_frames.append(chosen)
    summary_rows.append({"setting": setting, "all": len(frame), "coverage": len(coverage), "selected": len(chosen), "beam_GeV": ebeam, "median_epsilon_selected": chosen["epsilon"].median(), "charge_uC": total_charge_uC})

data = all_data[REFERENCE_KINEMATIC]
data_pre = pd.concat(coverage_frames, ignore_index=True)
data_selected = pd.concat(selected_frames, ignore_index=True)
if data_selected.empty:
    raise ValueError("No data remain after the chosen phase-space cuts")
display(pd.DataFrame(summary_rows))
print("All plots and bin candidates below use data; no simulation has been opened.")

,new_kin,ebeam_gev,hms_p_gev,hms_theta_deg,nps_theta_deg,nps_target_distance_cm,csv_source
KinC_x36_4,x36_q40_p4,8.455,2.562,24.78,10.1,357.0,/w/hallc-scshelf2102/nps/singhav/nps_analysis/pi0_analysis/root_analysis_env_main/config/nps_simulation_kinematics.csv
KinC_x36_5_407,x36_q40_p5,10.538,4.637,16.44,12.2,407.0,/w/hallc-scshelf2102/nps/singhav/nps_analysis/pi0_analysis/root_analysis_env_main/config/nps_simulation_kinematics_x36_5_407.csv


,setting,all,coverage,selected,beam_GeV,median_epsilon_selected,charge_uC
0,KinC_x36_4,76242,9885,9828,8.455,0.514739,880666.500
1,KinC_x36_5_407,108545,12193,11699,10.538,0.709886,382512.625


All plots and bin candidates below use data; no simulation has been opened.


## 2. Linked phase-space and binning dashboard

The colored occupancy background uses all **currently selected** events and updates after each cut. For the default pair, blue/orange means one setting and purple means both have at least `MIN_EVENTS_PER_CELL` events in that grid cell. 2D hover boxes and individual event dots are hidden initially so the cursor and shared cells stay clear. Use `Cell counts on hover` when you want local occupancy numbers; turn it off while placing corners. When shown, dots may be sampled for speed; the colored map, counts, bin edges, and occupancy use every selected event. Adjust `Cell min` to require more or fewer events from each setting per colored map cell. The four diamond corners are stored in $(x_B,Q^2)$, and the $W$ projection uses the proton-mass relation checked against the ROOT `W` branch.

In [4]:
# The linked dashboard is constructed after the data-only t-prime quantile helper is defined.

## 3. Data-only $t'$ edge candidates

This table starts from the configuration cell. The dashboard below recomputes uniform, count-quantile, or data-yield-quantile edges whenever the cuts change. The saved edge arrays and event counts in later cells use the latest dashboard selection when those cells are run.

In [5]:
def weighted_quantile(values, probabilities, weights=None):
    values = np.asarray(values, dtype=float)
    probabilities = np.asarray(probabilities, dtype=float)
    if weights is None:
        return np.quantile(values[np.isfinite(values)], probabilities)
    weights = np.asarray(weights, dtype=float)
    keep = np.isfinite(values) & np.isfinite(weights) & (weights > 0.0)
    values, weights = values[keep], weights[keep]
    if values.size == 0:
        raise ValueError("No positive-weight finite values for weighted quantiles")
    order = np.argsort(values)
    values, weights = values[order], weights[order]
    cumulative = np.cumsum(weights) - 0.5 * weights
    cumulative /= weights.sum()
    return np.interp(probabilities, cumulative, values)

def quantile_edges(values, n_bins, limits, weights=None):
    lo, hi = map(float, limits)
    inside = np.isfinite(values) & (np.asarray(values) >= lo) & (np.asarray(values) <= hi)
    selected_values = np.asarray(values)[inside]
    selected_weights = None if weights is None else np.asarray(weights)[inside]
    interior = weighted_quantile(selected_values, np.arange(1, n_bins) / n_bins, selected_weights)
    edges = np.r_[lo, interior, hi]
    if np.any(np.diff(edges) <= 0.0):
        raise ValueError(f"Collapsed quantile edges: {edges}")
    return edges

tprime_candidates = {
    "uniform": np.linspace(*TPRIME_LIMITS, N_TPRIME + 1),
    "count_quantile": quantile_edges(data_selected["tprime"], N_TPRIME, TPRIME_LIMITS),
    "yield_quantile": quantile_edges(data_selected["tprime"], N_TPRIME, TPRIME_LIMITS, data_selected["data_weight"]),
}
display(pd.DataFrame({name: [np.round(edges, 6).tolist()] for name, edges in tprime_candidates.items()}))

if TPRIME_EDGE_MODE == "manual":
    if MANUAL_TPRIME_EDGES is None:
        raise ValueError("Set MANUAL_TPRIME_EDGES when TPRIME_EDGE_MODE='manual'")
    TPRIME_EDGES = np.asarray(MANUAL_TPRIME_EDGES, dtype=float)
elif TPRIME_EDGE_MODE in tprime_candidates:
    TPRIME_EDGES = tprime_candidates[TPRIME_EDGE_MODE].copy()
else:
    raise ValueError(f"Unknown TPRIME_EDGE_MODE: {TPRIME_EDGE_MODE}")

if len(TPRIME_EDGES) != N_TPRIME + 1 or np.any(np.diff(TPRIME_EDGES) <= 0.0):
    raise ValueError("TPRIME_EDGES must contain N_TPRIME + 1 strictly increasing edges")
if not np.allclose([TPRIME_EDGES[0], TPRIME_EDGES[-1]], TPRIME_LIMITS):
    raise ValueError("TPRIME_EDGES endpoints must equal TPRIME_LIMITS")

TPRIME_CENTERS = 0.5 * (TPRIME_EDGES[:-1] + TPRIME_EDGES[1:])
PHI_CENTERS = 0.5 * (PHI_EDGES[:-1] + PHI_EDGES[1:])
print(f"Chosen data-only t' mode: {TPRIME_EDGE_MODE}")
print("t' edges [GeV^2]:", np.round(TPRIME_EDGES, 6))
print("t' centers [GeV^2]:", np.round(TPRIME_CENTERS, 6))
print("phi centers [deg]:", np.round(np.degrees(PHI_CENTERS), 1))

,uniform,count_quantile,yield_quantile
0,"[-0.7, -0.525, -0.35, -0.175, 0.0]","[-0.7, -0.289767, -0.185069, -0.103884, 0.0]","[-0.7, -0.289734, -0.182048, -0.100231, 0.0]"


Chosen data-only t' mode: count_quantile
t' edges [GeV^2]: [-0.7      -0.289767 -0.185069 -0.103884  0.      ]
t' centers [GeV^2]: [-0.494884 -0.237418 -0.144477 -0.051942]
phi centers [deg]: [ 10.  30.  50.  70.  90. 110. 130. 150. 170. 190. 210. 230. 250. 270.
 290. 310. 330. 350.]


In [6]:
# Create the linked dashboard once. Rerun this cell to reset its controls to the configuration cell.
import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.xsec_extract.xsec_bin_dashboard import BinDashboard

def receive_dashboard_selection(view):
    global data_selected, selected_frames, TPRIME_LIMITS, Q2_LIMITS, XB_LIMITS
    global DIAMOND_XB_Q2_VERTICES, N_TPRIME, N_Q2, N_XB_PER_Q2, N_PHI
    global TPRIME_EDGE_MODE, Q2_EDGE_MODE, XB_EDGE_MODE, PHI_EDGES
    global TPRIME_EDGES, TPRIME_CENTERS, PHI_CENTERS
    selected_frames = view.selected_frames
    data_selected = pd.concat(selected_frames, ignore_index=True)
    TPRIME_LIMITS = tuple(view.t_limits.value)
    Q2_LIMITS = tuple(view.q_limits.value)
    XB_LIMITS = tuple(view.x_limits.value)
    polygon = view._polygon()
    DIAMOND_XB_Q2_VERTICES = None if polygon is None else [tuple(map(float, v)) for v in polygon]
    N_TPRIME, N_Q2, N_XB_PER_Q2, N_PHI = view.n_t.value, view.n_q.value, view.n_x.value, view.n_phi.value
    TPRIME_EDGE_MODE, Q2_EDGE_MODE, XB_EDGE_MODE = view.t_mode.value, view.q_mode.value, view.x_mode.value
    TPRIME_EDGES = view.edges["tprime"]
    TPRIME_CENTERS = (TPRIME_EDGES[:-1] + TPRIME_EDGES[1:]) / 2
    PHI_EDGES = view.edges["phi"]
    PHI_CENTERS = (PHI_EDGES[:-1] + PHI_EDGES[1:]) / 2

dashboard = BinDashboard(
    settings=KINEMATICS, coverage_frames=coverage_frames, proton_mass=PROTON_MASS_GEV,
    limits={"tprime": TPRIME_LIMITS, "Q2": Q2_LIMITS, "xB": XB_LIMITS},
    modes={"tprime": TPRIME_EDGE_MODE, "Q2": Q2_EDGE_MODE, "xB": XB_EDGE_MODE},
    counts={"tprime": N_TPRIME, "Q2": N_Q2, "xB": N_XB_PER_Q2, "phi": N_PHI},
    vertices=DIAMOND_XB_Q2_VERTICES,
    manual_edges={"tprime": MANUAL_TPRIME_EDGES, "Q2": MANUAL_Q2_EDGES, "xB": XB_EDGES_BY_Q2},
    on_change=receive_dashboard_selection,
    overlap_bins=OVERLAP_BINS, min_events_per_cell=MIN_EVENTS_PER_CELL,
    max_scatter=MAX_SCATTER_PER_SETTING,
)
dashboard.show()

## 4. Inspect proposed $Q^2$ and $x_B$ edges

The linked dashboard above shows the current data distributions and conditional bin lines. The candidate table below lists uniform and quantile alternatives using the current selection. Run it again after changing dashboard controls.

In [7]:
# Candidate edges from the selected data; all outer edges are user-specified limits.
candidate_rows = []
for variable, limits in [("Q2", Q2_LIMITS), ("xB", XB_LIMITS)]:
    for n_bins in PREVIEW_BIN_COUNTS:
        for method, weights in [("uniform", "uniform"), ("count_quantile", None), ("yield_quantile", data_selected["data_weight"])]:
            edges = np.linspace(*limits, n_bins + 1) if method == "uniform" else quantile_edges(data_selected[variable], n_bins, limits, weights)
            candidate_rows.append({"variable": variable, "method": method, "n_bins": n_bins, "edges": np.round(edges, 6).tolist()})
display(pd.DataFrame(candidate_rows))

preview_q2_edges = quantile_edges(data_selected["Q2"], PREVIEW_N_Q2, Q2_LIMITS)
conditional_rows = []
for iq in range(PREVIEW_N_Q2):
    last = iq == PREVIEW_N_Q2 - 1
    qmask = (data_selected["Q2"] >= preview_q2_edges[iq]) & (
        data_selected["Q2"] <= preview_q2_edges[iq + 1] if last else data_selected["Q2"] < preview_q2_edges[iq + 1]
    )
    xb_edges = quantile_edges(data_selected.loc[qmask, "xB"], PREVIEW_N_XB_PER_Q2, XB_LIMITS)
    conditional_rows.append({"q2_bin": iq, "q2_edges": np.round(preview_q2_edges[[iq, iq + 1]], 6).tolist(),
        "data_events": int(qmask.sum()), "xB_count_quantile_edges": np.round(xb_edges, 6).tolist()})
display(Markdown("**Data-only conditional preview:**"))
display(pd.DataFrame(conditional_rows))

,variable,method,n_bins,edges
0,Q2,uniform,1,"[3.0, 5.0]"
1,Q2,count_quantile,1,"[3.0, 5.0]"
2,Q2,yield_quantile,1,"[3.0, 5.0]"
3,Q2,uniform,2,"[3.0, 4.0, 5.0]"
4,Q2,count_quantile,2,"[3.0, 3.906249, 5.0]"
5,Q2,yield_quantile,2,"[3.0, 3.894737, 5.0]"
6,Q2,uniform,3,"[3.0, 3.666667, 4.333333, 5.0]"
7,Q2,count_quantile,3,"[3.0, 3.727394, 4.096391, 5.0]"
8,Q2,yield_quantile,3,"[3.0, 3.703921, 4.095076, 5.0]"
9,Q2,uniform,4,"[3.0, 3.5, 4.0, 4.5, 5.0]"


**Data-only conditional preview:**

,q2_bin,q2_edges,data_events,xB_count_quantile_edges
0,0,"[3.0, 3.906249]",10763,"[0.25, 0.323669, 0.455]"
1,1,"[3.906249, 5.0]",10764,"[0.25, 0.384834, 0.455]"


## 5. Review $Q^2$/$x_B$ edges from the current data selection

The dashboard updates the selected events and configured edge modes in this kernel. Run the cell below after changing cuts or bins to print exact edges and per-setting occupancy. Manual arrays in the configuration cell remain available for finalized export.

In [8]:
def proposed_edges(values, limits, n_bins, mode, manual=None, weights=None):
    if mode == "manual":
        if manual is None:
            raise ValueError("Manual edge mode requires an explicit edge array")
        edges = np.asarray(manual, dtype=float)
    elif mode == "uniform":
        edges = np.linspace(*limits, n_bins + 1)
    elif mode == "count_quantile":
        edges = quantile_edges(values, n_bins, limits)
    elif mode == "yield_quantile":
        edges = quantile_edges(values, n_bins, limits, weights)
    else:
        raise ValueError(f"Unknown edge mode: {mode}")
    if len(edges) < 2 or not np.all(np.isfinite(edges)) or np.any(np.diff(edges) <= 0):
        raise ValueError(f"Invalid edge array: {edges}")
    if not np.allclose([edges[0], edges[-1]], limits):
        raise ValueError(f"Edge endpoints must equal {limits}: {edges}")
    return edges

q2_edges_final = proposed_edges(data_selected["Q2"], Q2_LIMITS, N_Q2, Q2_EDGE_MODE, MANUAL_Q2_EDGES, data_selected["data_weight"])
if XB_EDGES_BY_Q2 is not None and len(XB_EDGES_BY_Q2) != len(q2_edges_final) - 1:
    raise ValueError("Provide one xB edge array per Q2 bin")
xb_edges_by_q2_final = []
for iq in range(len(q2_edges_final) - 1):
    qmask = ((data_selected["Q2"] >= q2_edges_final[iq]) &
        (data_selected["Q2"] <= q2_edges_final[iq + 1] if iq == len(q2_edges_final) - 2 else data_selected["Q2"] < q2_edges_final[iq + 1]))
    subset = data_selected.loc[qmask]
    manual_xb = None if XB_EDGES_BY_Q2 is None else XB_EDGES_BY_Q2[iq]
    mode = "manual" if manual_xb is not None else XB_EDGE_MODE
    xb_edges_by_q2_final.append(proposed_edges(subset["xB"], XB_LIMITS, N_XB_PER_Q2, mode, manual_xb, subset["data_weight"]))
Q2_XB_FINALIZED = Q2_EDGE_MODE == "manual" and (XB_EDGES_BY_Q2 is not None or XB_EDGE_MODE == "manual")
print("Q2 edges:", q2_edges_final)
for iq, edges in enumerate(xb_edges_by_q2_final):
    print(f"xB edges in Q2 bin {iq}: {edges}")
print("Final manual edges:" if Q2_XB_FINALIZED else "Preview edges; export disabled")

coverage_rows = []
for setting, frame in zip(KINEMATICS, selected_frames):
    for iq, xb_edges in enumerate(xb_edges_by_q2_final):
        qmask = (frame["Q2"] >= q2_edges_final[iq]) & (frame["Q2"] <= q2_edges_final[iq + 1] if iq == len(q2_edges_final) - 2 else frame["Q2"] < q2_edges_final[iq + 1])
        for ix in range(len(xb_edges) - 1):
            xmask = (frame["xB"] >= xb_edges[ix]) & (frame["xB"] <= xb_edges[ix + 1] if ix == len(xb_edges) - 2 else frame["xB"] < xb_edges[ix + 1])
            coverage_rows.append({"setting": setting, "q2_bin": iq, "xb_bin": ix, "events": int((qmask & xmask).sum())})
display(pd.DataFrame(coverage_rows))

Q2 edges: [3.         3.90624907 5.        ]
xB edges in Q2 bin 0: [0.25       0.32366883 0.455     ]
xB edges in Q2 bin 1: [0.25       0.38483385 0.455     ]
Preview edges; export disabled


,setting,q2_bin,xb_bin,events
0,KinC_x36_4,0,0,2050
1,KinC_x36_4,0,1,2676
2,KinC_x36_4,1,0,2757
3,KinC_x36_4,1,1,2345
4,KinC_x36_5_407,0,0,3331
5,KinC_x36_5_407,0,1,2706
6,KinC_x36_5_407,1,0,2625
7,KinC_x36_5_407,1,1,3037


## 6. SIMC-derived representative $Q^2$ and $x_B$ after binning is frozen

This is the first section that opens the simulation for `REFERENCE_KINEMATIC`. It cannot alter any edge. Reconstructed variables decide which accepted exclusive event belongs to a frozen bin. The requested representative values use `full_weight` and generated $Q_i^2$ plus $x_{B,i}=Q_i^2/(W_i^2-M_p^2+Q_i^2)$. Reconstructed weighted means are retained only as diagnostics. All $\phi$ bins are combined.

These are accepted-SIMC, model-weighted kinematic means; they are not efficiencies or acceptance corrections.

In [9]:
SIM_BRANCHES = [
    "Q2", "xB", "t", "tmin", "phi", "mmiss", "full_weight",
    "is_exclusive", "Q2i", "Wi",
]
if SIM_FILE is None or not SIM_FILE.is_file():
    sim_selected = pd.DataFrame(columns=SIM_BRANCHES + ["tprime", "phi_wrapped", "xBi"])
    print(f"No SIMC input configured for {REFERENCE_KINEMATIC}; representative coordinates unavailable")
else:
    sim_root, sim_tree = open_checked_tree(SIM_FILE, SIM_TREE, SIM_BRANCHES)
    sim = tree_frame(sim_tree, SIM_BRANCHES)
    sim_root.close()
    sim["tprime"] = sim["t"] - sim["tmin"]
    sim["phi_wrapped"] = np.mod(sim["phi"], 2.0 * np.pi)
    xb_i_denominator = sim["Wi"] ** 2 - PROTON_MASS_GEV ** 2 + sim["Q2i"]
    sim["xBi"] = np.where(xb_i_denominator > 0.0, sim["Q2i"] / xb_i_denominator, np.nan)
    sim_finite = np.logical_and.reduce([np.isfinite(sim[name]) for name in ["Q2", "xB", "tprime", "phi_wrapped", "mmiss", "full_weight"]])
    sim_selected_mask = (
        sim_finite & (sim["is_exclusive"] != 0) & (sim["full_weight"] >= 0.0)
        & sim["mmiss"].between(*MMISS_LIMITS, inclusive="both")
        & sim["tprime"].between(TPRIME_EDGES[0], TPRIME_EDGES[-1], inclusive="both")
        & sim["Q2"].between(q2_edges_final[0], q2_edges_final[-1], inclusive="both")
        & sim["xB"].between(*XB_LIMITS, inclusive="both") & diamond_mask(sim)
    )
    sim_selected = sim.loc[sim_selected_mask].copy()

def weighted_mean(values, weights):
    values, weights = np.asarray(values, dtype=float), np.asarray(weights, dtype=float)
    keep = np.isfinite(values) & np.isfinite(weights) & (weights > 0.0)
    return np.average(values[keep], weights=weights[keep]) if np.any(keep) else np.nan

def in_bin(values, edges, index):
    upper = values <= edges[index + 1] if index == len(edges) - 2 else values < edges[index + 1]
    return (values >= edges[index]) & upper

average_rows = []
for it in range(N_TPRIME):
    tmask_sim = in_bin(sim_selected["tprime"], TPRIME_EDGES, it)
    reference_data = data_selected.loc[data_selected["setting"] == REFERENCE_KINEMATIC]
    tmask_data = in_bin(reference_data["tprime"], TPRIME_EDGES, it)
    for iq in range(len(q2_edges_final) - 1):
        qmask_sim = in_bin(sim_selected["Q2"], q2_edges_final, iq)
        qmask_data = in_bin(reference_data["Q2"], q2_edges_final, iq)
        xb_edges = xb_edges_by_q2_final[iq]
        for ix in range(len(xb_edges) - 1):
            mask_sim = tmask_sim & qmask_sim & in_bin(sim_selected["xB"], xb_edges, ix)
            mask_data = tmask_data & qmask_data & in_bin(reference_data["xB"], xb_edges, ix)
            accepted = sim_selected.loc[mask_sim]
            weights = accepted["full_weight"].to_numpy()
            sumw = weights.sum()
            sumw2 = np.square(weights).sum()
            average_rows.append({
                "tprime_bin": it, "q2_bin": iq, "xb_bin": ix,
                "tprime_center": TPRIME_CENTERS[it],
                "data_events_phi_integrated": int(mask_data.sum()),
                "accepted_sim_events_phi_integrated": int(mask_sim.sum()),
                "accepted_sim_neff": (sumw * sumw / sumw2) if sumw2 > 0.0 else 0.0,
                "Q2_acceptance_weighted": weighted_mean(accepted["Q2i"], weights),
                "xB_acceptance_weighted": weighted_mean(accepted["xBi"], weights),
                "Q2_reco_weighted_diagnostic": weighted_mean(accepted["Q2"], weights),
                "xB_reco_weighted_diagnostic": weighted_mean(accepted["xB"], weights),
            })
acceptance_weighted_table = pd.DataFrame(average_rows)
print(f"Accepted exclusive SIMC entries in configured window: {len(sim_selected):,}")
display(acceptance_weighted_table.round(6))

Accepted exclusive SIMC entries in configured window: 141,931


,tprime_bin,q2_bin,xb_bin,tprime_center,data_events_phi_integrated,accepted_sim_events_phi_integrated,accepted_sim_neff,Q2_acceptance_weighted,xB_acceptance_weighted,Q2_reco_weighted_diagnostic,xB_reco_weighted_diagnostic
0,0,0,0,-0.494884,760,9963,8478.725586,3.467113,0.302173,3.454267,0.298850
1,0,0,1,-0.494884,536,5751,5107.242188,3.768083,0.347015,3.761809,0.344225
2,0,1,0,-0.494884,634,9608,8577.562500,4.098044,0.367366,4.092820,0.364411
3,0,1,1,-0.494884,973,13382,12498.985352,4.444789,0.419038,4.444102,0.416510
4,1,0,0,-0.237418,782,7710,6121.875488,3.459753,0.303473,3.448267,0.300490
5,1,0,1,-0.237418,671,6246,5519.791016,3.753502,0.347356,3.747522,0.344598
6,1,1,0,-0.237418,628,8380,7434.766113,4.096367,0.367790,4.091431,0.364883
7,1,1,1,-0.237418,762,9747,9287.081055,4.426104,0.416226,4.423329,0.413431
8,2,0,0,-0.144477,901,7826,6422.593262,3.458530,0.303941,3.447565,0.301002
9,2,0,1,-0.144477,737,7129,6380.296387,3.746788,0.346691,3.742174,0.344406


## 7. Review coordinates and optionally export

The $t'$ and $\phi$ coordinates below are geometric centers by construction. The $Q^2$/$x_B$ table above is integrated over all $\phi$ bins. Export is permitted only after explicit data-chosen $Q^2$ and $x_B$ edges have been entered.

In [10]:
coordinate_table = pd.DataFrame({
    "phi_bin": np.arange(N_PHI),
    "phi_lo_rad": PHI_EDGES[:-1],
    "phi_hi_rad": PHI_EDGES[1:],
    "phi_center_rad": PHI_CENTERS,
    "phi_center_deg": np.degrees(PHI_CENTERS),
})
display(Markdown("**Fixed phi coordinates**"))
display(coordinate_table.round(6))
display(Markdown("**Fixed t-prime coordinates**"))
display(pd.DataFrame({
    "tprime_bin": np.arange(N_TPRIME),
    "tprime_lo": TPRIME_EDGES[:-1],
    "tprime_hi": TPRIME_EDGES[1:],
    "tprime_center": TPRIME_CENTERS,
}).round(6))

def json_records(frame):
    cleaned = frame.astype(object).where(pd.notnull(frame), None)
    return cleaned.to_dict(orient="records")

bin_contract = {
    "schema_version": 1,
    "kinematic": REFERENCE_KINEMATIC,
    "settings_compared": KINEMATICS,
    "data_files": {key: str(DATA_FILES[key]) for key in KINEMATICS},
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "bin selection and representative coordinates only; no cross sections",
    "data_file": str(DATA_FILES[REFERENCE_KINEMATIC]),
    "data_tree": DATA_TREE,
    "sim_file": str(SIM_FILE) if SIM_FILE is not None else None,
    "sim_tree": SIM_TREE,
    "edge_provenance": "data_only",
    "tprime_definition": "t - tmin",
    "phi_definition": "phi modulo 2*pi",
    "selection": {
        "mmiss_gev": list(MMISS_LIMITS), "tprime_gev2": list(TPRIME_LIMITS),
        "q2_gev2": list(Q2_LIMITS), "xb": list(XB_LIMITS),
        "diamond_xb_q2_vertices": DIAMOND_XB_Q2_VERTICES,
    },
    "tprime_edge_mode": TPRIME_EDGE_MODE,
    "tprime_edges_gev2": TPRIME_EDGES.tolist(),
    "tprime_centers_gev2": TPRIME_CENTERS.tolist(),
    "phi_edges_rad": PHI_EDGES.tolist(),
    "phi_centers_rad": PHI_CENTERS.tolist(),
    "q2_edges_gev2": q2_edges_final.tolist(),
    "xb_edges_by_q2": [edges.tolist() for edges in xb_edges_by_q2_final],
    "q2_xb_edges_finalized": bool(Q2_XB_FINALIZED),
    "representative_q2_xb": {
        "source": "accepted generated-exclusive SIMC after reconstructed-bin selection",
        "weight": "full_weight",
        "phi_integration": f"all {N_PHI} phi bins together",
        "q2_variable": "Q2i",
        "xb_variable": "Q2i / (Wi^2 - Mp^2 + Q2i)",
        "rows": json_records(acceptance_weighted_table),
    },
}

print(json.dumps({k: v for k, v in bin_contract.items() if k != "representative_q2_xb"}, indent=2))
if WRITE_EXPORT:
    if len(KINEMATICS) != 1 or SIM_FILE is None or not SIM_FILE.is_file():
        raise RuntimeError("Export requires one setting and its configured SIMC input")
    if not Q2_XB_FINALIZED:
        raise RuntimeError("Refusing export: set data-chosen Q2_EDGES and XB_EDGES_BY_Q2 first")
    EXPORT_PATH.write_text(json.dumps(bin_contract, indent=2) + "\n", encoding="utf-8")
    print(f"Wrote {EXPORT_PATH}")
else:
    print("Preview only. Set WRITE_EXPORT=True after every edge is final.")

**Fixed phi coordinates**

,phi_bin,phi_lo_rad,phi_hi_rad,phi_center_rad,phi_center_deg
0,0,0.000000,0.349066,0.174533,10.0
1,1,0.349066,0.698132,0.523599,30.0
2,2,0.698132,1.047198,0.872665,50.0
3,3,1.047198,1.396263,1.221730,70.0
4,4,1.396263,1.745329,1.570796,90.0
5,5,1.745329,2.094395,1.919862,110.0
6,6,2.094395,2.443461,2.268928,130.0
7,7,2.443461,2.792527,2.617994,150.0
8,8,2.792527,3.141593,2.967060,170.0
9,9,3.141593,3.490659,3.316126,190.0


**Fixed t-prime coordinates**

,tprime_bin,tprime_lo,tprime_hi,tprime_center
0,0,-0.700000,-0.289767,-0.494884
1,1,-0.289767,-0.185069,-0.237418
2,2,-0.185069,-0.103884,-0.144477
3,3,-0.103884,0.000000,-0.051942


{
  "schema_version": 1,
  "kinematic": "KinC_x36_5_407",
  "settings_compared": [
    "KinC_x36_4",
    "KinC_x36_5_407"
  ],
  "data_files": {
    "KinC_x36_4": "/w/hallc-scshelf2102/nps/singhav/nps_analysis/pi0_analysis/root_analysis_env_main/output/KinC_x36_4/root/combined_branches_LH2.root",
    "KinC_x36_5_407": "/w/hallc-scshelf2102/nps/singhav/nps_analysis/pi0_analysis/root_analysis_env_main/output/KinC_x36_5_407/KinC_x36_5/root/combined_branches_LH2.root"
  },
  "created_utc": "2026-09-27T18:33:49.024852+00:00",
  "purpose": "bin selection and representative coordinates only; no cross sections",
  "data_file": "/w/hallc-scshelf2102/nps/singhav/nps_analysis/pi0_analysis/root_analysis_env_main/output/KinC_x36_5_407/KinC_x36_5/root/combined_branches_LH2.root",
  "data_tree": "physics",
  "sim_file": "/volatile/hallc/nps/singhav/nps_smearing/smear_x36_5_407/smearing_output/KinC_x36_5_407/root/simc_pi0_analysis_output_smeared.root",
  "sim_tree": "simulation",
  "edge_provenance": 